# Table 5.4 — Cold-storage test set metrics by fog level

依 `plot/test_metadata.json` 的 dataloader 序號與 `fog_level`（low / medium / heavy），彙總各方法的 **PSNR / SSIM**。

- 優先讀取實驗目錄下 `logs/per_image_metrics.json`（與 `infer.py` 一致）；**DCP** 等外部基線可直接指定 `metrics_path` 讀 JSON。
- 若無 JSON，則從 `results/{step}_{index}_out.png` 對 GT 重算。
- **Foggy input** 基線：以任一已完成推理目錄中的 `{step}_{index}_lr.png` 對 GT 計算（與模型輸入解析度一致）；結果快取至 `plot/data/foggy_input_metrics_cache.json`，下次執行直接讀取。

在 **Methods 設定** 儲存格填入各方法對應的 `exp_dir`；尚未跑完推理的方法可留 `None`，表格會顯示 `—`。

In [3]:
import json
import os
import sys
from collections import Counter, defaultdict
from os import path as osp
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

ROOT = Path("/mnt/newdisk/Documents/linzhanyang/DehazeDDPM").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core import metrics as Metrics

# --- paths (edit as needed) ---
METADATA_PATH = ROOT / "plot/test_metadata.json"
INFER_SAVE_STEP = 0
# Used only for Foggy-input baseline (_lr.png source); any completed infer run is fine.
FOGGY_LR_EXP_DIR = ROOT / (
    "experiments/test/physical_v1/"
    "Dehaze_ColdFog_finetune_test_ddim100_physical_v1_260526_091235"
)
FOGGY_CACHE_PATH = ROOT / "plot/data/foggy_input_metrics_cache.json"
DCP_METRICS_PATH = Path(
    "/mnt/newdisk/Documents/linzhanyang/DCP/results/dehazeddpm_test/logs/per_image_metrics.json"
)

FOG_SHORT = {"low": "L", "medium": "M", "heavy": "H"}
FOG_ORDER = ("low", "medium", "heavy")
METRIC_COLS = [
    "L PSNR", "L SSIM", "M PSNR", "M SSIM", "H PSNR", "H SSIM",
    "Avg PSNR", "Avg SSIM",
]

In [4]:
from __future__ import annotations

def load_metadata(path: Path):
    with open(path, encoding="utf-8") as f:
        meta = json.load(f)
    index_to_fog = {s["index"]: s["fog_level"] for s in meta["samples"]}
    index_to_filename = {s["index"]: s["filename"] for s in meta["samples"]}
    return meta, index_to_fog, index_to_filename, meta["dataroothq"]


def load_rgb(path: str) -> np.ndarray:
    return np.asarray(Image.open(path).convert("RGB"))


def load_gt_resized(gt_root: str, filename: str, target_hw) -> np.ndarray:
    h, w = target_hw
    gt_path = osp.join(gt_root, filename)
    im = Image.open(gt_path).convert("RGB").resize((w, h), Image.LANCZOS)
    return np.asarray(im)


def load_per_image_metrics(metrics_path: str) -> dict[int, dict]:
    with open(metrics_path, encoding="utf-8") as f:
        data = json.load(f)
    return {int(r["index"]): r for r in data["per_image"]}


def compute_metrics_from_results(
    exp_dir,
    gt_root: str,
    index_to_filename: dict[int, str],
    *,
    mode: str = "output",
    infer_step: int = 0,
) -> dict[int, dict]:
    """mode='output' -> _out.png vs GT; mode='foggy' -> _lr.png vs GT."""
    results_dir = osp.join(str(exp_dir), "results")
    suffix = "out" if mode == "output" else "lr"
    records = {}
    for idx, filename in sorted(index_to_filename.items()):
        img_path = osp.join(results_dir, f"{infer_step}_{idx}_{suffix}.png")
        if not osp.isfile(img_path):
            raise FileNotFoundError(img_path)
        pred = load_rgb(img_path)
        gt = load_gt_resized(gt_root, filename, pred.shape[:2])
        records[idx] = {
            "index": idx,
            "psnr": float(Metrics.calculate_psnr(pred, gt)),
            "ssim": float(Metrics.calculate_ssim(pred, gt)),
        }
    return records


def resolve_per_image_metrics(
    exp_dir,
    gt_root: str,
    index_to_filename: dict[int, str],
    *,
    mode: str = "output",
    infer_step: int = 0,
) -> dict[int, dict]:
    metrics_path = osp.join(str(exp_dir), "logs", "per_image_metrics.json")
    if mode == "output" and osp.isfile(metrics_path):
        return load_per_image_metrics(metrics_path)
    return compute_metrics_from_results(
        exp_dir, gt_root, index_to_filename, mode=mode, infer_step=infer_step
    )


def _foggy_cache_key(
    metadata_path: Path,
    exp_dir: Path,
    gt_root: str,
    index_to_filename: dict[int, str],
    infer_step: int,
) -> dict:
    return {
        "metadata_path": str(metadata_path.resolve()),
        "sample_count": len(index_to_filename),
        "gt_root": gt_root,
        "exp_dir": str(exp_dir.resolve()),
        "infer_step": infer_step,
    }


def _load_foggy_cache(cache_path: Path, cache_key: dict) -> dict[int, dict] | None:
    if not cache_path.is_file():
        return None
    with open(cache_path, encoding="utf-8") as f:
        payload = json.load(f)
    if payload.get("cache_key") != cache_key:
        return None
    return {int(r["index"]): r for r in payload["per_image"]}


def _save_foggy_cache(
    cache_path: Path,
    cache_key: dict,
    per_image: dict[int, dict],
) -> None:
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "cache_key": cache_key,
        "per_image": [per_image[idx] for idx in sorted(per_image)],
    }
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def resolve_foggy_metrics(
    exp_dir,
    gt_root: str,
    index_to_filename: dict[int, str],
    *,
    metadata_path: Path,
    cache_path: Path,
    infer_step: int = 0,
    force_recompute: bool = False,
) -> dict[int, dict]:
    cache_key = _foggy_cache_key(
        metadata_path, Path(exp_dir), gt_root, index_to_filename, infer_step
    )
    if not force_recompute:
        cached = _load_foggy_cache(cache_path, cache_key)
        if cached is not None:
            print(f"  (foggy metrics loaded from cache: {cache_path.name})")
            return cached

    per_image = compute_metrics_from_results(
        exp_dir, gt_root, index_to_filename, mode="foggy", infer_step=infer_step
    )
    _save_foggy_cache(cache_path, cache_key, per_image)
    print(f"  (foggy metrics saved to {cache_path.name})")
    return per_image


def aggregate_by_fog(per_image: dict[int, dict], index_to_fog: dict[int, str]) -> dict:
    groups = defaultdict(list)
    for idx, rec in per_image.items():
        groups[index_to_fog[idx]].append(rec)

    row = {}
    all_psnr, all_ssim = [], []
    for fog in FOG_ORDER:
        short = FOG_SHORT[fog]
        items = groups.get(fog, [])
        if items:
            row[f"{short} PSNR"] = float(np.mean([x["psnr"] for x in items]))
            row[f"{short} SSIM"] = float(np.mean([x["ssim"] for x in items]))
            all_psnr.extend(x["psnr"] for x in items)
            all_ssim.extend(x["ssim"] for x in items)
        else:
            row[f"{short} PSNR"] = np.nan
            row[f"{short} SSIM"] = np.nan

    row["Avg PSNR"] = float(np.mean(all_psnr)) if all_psnr else np.nan
    row["Avg SSIM"] = float(np.mean(all_ssim)) if all_ssim else np.nan
    return row


def format_table(df: pd.DataFrame, digits: int = 2) -> pd.DataFrame:
    out = df.copy()
    for col in METRIC_COLS:
        if col in out.columns:
            out[col] = out[col].apply(
                lambda x: "—" if pd.isna(x) else f"{float(x):.{digits}f}"
            )
    return out


meta, index_to_fog, index_to_filename, gt_root = load_metadata(METADATA_PATH)
print(f"Samples: {meta['count']}  |  fog counts: {dict(Counter(index_to_fog.values()))}")

Samples: 80  |  fog counts: {'medium': 27, 'low': 27, 'heavy': 26}


In [5]:
# Table 5.4 rows — set exp_dir to experiment root (contains logs/ and results/)
# Leave exp_dir=None until infer finished; metrics cells will show "—".

METHODS = [
    {
        "method": "Foggy input",
        "checkpoint": "—",
        "sampler": "—",
        "source": "foggy",
    },
    {
        "method": "DCP",
        "checkpoint": "—",
        "sampler": "—",
        "metrics_path": DCP_METRICS_PATH,
    },
    {
        "method": "DehazeDDPM-DENSE pre-train",
        "checkpoint": "DENSE_I130000_E260",
        "sampler": "DDIM100",
        "exp_dir": ROOT / (
            "experiments/test/pretrain_model_domain_gap/"
            "Dehaze_ColdFog_pretrain_test_DENSE_ddim100_260526_110402"
        ),
    },
    {
        "method": "DehazeDDPM-NH pre-train",
        "checkpoint": "NH_I230000_E460",
        "sampler": "DDIM100",
        "exp_dir": ROOT / (
            "experiments/test/pretrain_model_domain_gap/"
            "Dehaze_ColdFog_pretrain_test_NH_ddim100_260526_111436"
        ),
    },
    {
        "method": "netG-only",
        "checkpoint": "I85000_E304",
        "sampler": "DDIM100",
        "exp_dir": ROOT / (
            "experiments/test/netG_only/"
            "Dehaze_ColdFog_finetune_test_ddim100_netG_260526_172111"
        ),
    },
    {
        "method": "netG+netH (l_pix)",
        "checkpoint": "I90000_E322",
        "sampler": "DDIM100",
        "exp_dir": ROOT / (
            "experiments/test/netG_H_l_pix/"
            "Dehaze_ColdFog_finetune_test_ddim100_netG_H_lpix_260526_171655"
        ),
    },
    {
        "method": "PPDM",
        "checkpoint": "I75000_E268",
        "sampler": "DDIM100",
        # 物理 loss 版本（lambda_t + lambda_asm）
        "exp_dir": ROOT / (
            "experiments/test/physical_v1/"
            "Dehaze_ColdFog_finetune_test_ddim100_physical_v1_260526_091235"
        ),
    },
]

In [6]:
rows = []

for entry in METHODS:
    row = {
        "Method": entry["method"],
        "Checkpoint": entry["checkpoint"],
        "Sampler": entry["sampler"],
    }

    try:
        if entry.get("source") == "foggy":
            per_image = resolve_foggy_metrics(
                FOGGY_LR_EXP_DIR,
                gt_root,
                index_to_filename,
                metadata_path=METADATA_PATH,
                cache_path=FOGGY_CACHE_PATH,
                infer_step=INFER_SAVE_STEP,
            )
        elif entry.get("metrics_path"):
            per_image = load_per_image_metrics(str(entry["metrics_path"]))
        elif entry.get("exp_dir"):
            per_image = resolve_per_image_metrics(
                entry["exp_dir"],
                gt_root,
                index_to_filename,
                mode="output",
                infer_step=INFER_SAVE_STEP,
            )
        else:
            per_image = None

        if per_image is not None:
            stats = aggregate_by_fog(per_image, index_to_fog)
            row.update(stats)
            print(f"OK  {entry['method']:32s}  n={len(per_image)}  Avg PSNR={stats['Avg PSNR']:.2f}")
        else:
            for col in METRIC_COLS:
                row[col] = np.nan
            print(f"SKIP {entry['method']:32s}  (exp_dir not set)")
    except FileNotFoundError as exc:
        for col in METRIC_COLS:
            row[col] = np.nan
        print(f"MISS {entry['method']:32s}  {exc}")

    rows.append(row)

raw_df = pd.DataFrame(rows)
display_df = format_table(raw_df)
display(display_df)

  (foggy metrics loaded from cache: foggy_input_metrics_cache.json)
OK  Foggy input                       n=80  Avg PSNR=13.90
OK  DCP                               n=80  Avg PSNR=16.30
OK  DehazeDDPM-DENSE pre-train        n=80  Avg PSNR=14.87
OK  DehazeDDPM-NH pre-train           n=80  Avg PSNR=14.73
OK  netG-only                         n=80  Avg PSNR=17.79
OK  netG+netH (l_pix)                 n=80  Avg PSNR=18.73
OK  PPDM                              n=80  Avg PSNR=21.52


,Method,Checkpoint,Sampler,L PSNR,L SSIM,M PSNR,M SSIM,H PSNR,H SSIM,Avg PSNR,Avg SSIM
0,Foggy input,—,—,20.10,0.90,12.24,0.71,9.18,0.54,13.90,0.72
1,DCP,—,—,15.97,0.88,17.36,0.87,15.53,0.78,16.30,0.84
2,DehazeDDPM-DENSE pre-train,DENSE_I130000_E260,DDIM100,16.26,0.55,14.79,0.59,13.52,0.52,14.87,0.55
3,DehazeDDPM-NH pre-train,NH_I230000_E460,DDIM100,17.08,0.63,14.16,0.53,12.87,0.40,14.73,0.52
4,netG-only,I85000_E304,DDIM100,19.33,0.71,17.95,0.69,16.03,0.59,17.79,0.66
5,netG+netH (l_pix),I90000_E322,DDIM100,20.65,0.89,18.61,0.84,16.87,0.77,18.73,0.83
6,PPDM,I75000_E268,DDIM100,24.38,0.91,21.45,0.86,18.61,0.76,21.52,0.84
